# MediSaathi: TrOCR 32-image sanity experiment
Infrastructure only. No cell has been executed. Opening or running the defaults does not authorize downloads, inference, training, or uploads.
Use a private notebook and private data source. Audit selected images and establish cloud-upload permission first. Never attach Test_Set, .env, keys, .git, or the repository.
Dataset: RxHandBD: A Handwritten Prescription Word Image Dataset; Md Masudul Islam; version 3; DOI [10.17632/dsb5r6vskg.3](https://data.mendeley.com/datasets/dsb5r6vskg/3); [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Derived duplicate-aware split and eight exclusions; labels unchanged. The license does not establish consent or de-identification.

## Environment and allowlisted source files
Target Python 3.12; PyTorch 2.8.0 with CUDA compatible with the allocated GPU/driver; Transformers 4.57.6; Pillow 12.0.0; NumPy 2.2.6; SentencePiece 0.2.1; protobuf 6.33.6. Do not install the Windows CPU wheel in the cloud. Inspect the runtime before separately approving installation. Record pip freeze privately. No installation cell runs automatically.
The helpers require trocr_training.py, trocr_baseline.py, and evaluate_ocr.py. Transfer only those allowlisted source files after approval. Do not upload the repository. Configure the cloud paths below without editing frozen manifests.

In [ ]:
from pathlib import Path
from trocr_training import preflight, train_sanity, TrainingConfig
SPLIT_MANIFEST = Path('/kaggle/input/medisaathi-approved/split_manifest.json')
BASELINE_MANIFEST = Path('/kaggle/input/medisaathi-approved/baseline100_manifest.json')
TRAIN_DIR = Path('/kaggle/input/medisaathi-approved/Train_Set')
OUTPUT_DIR = Path('/kaggle/working/medisaathi-sanity-run-001')
RUN_PREFLIGHT = False
RUN_TRAINING = False
CLOUD_PRIVACY_AND_UPLOAD_APPROVED = False


## Input preflight only
Verifies frozen hashes, training partition membership, exclusions, 32 distinct groups, and selected image hashes. No torch imports, model loading, tokenization, inference, or training. Enable only when the approved files exist.

In [ ]:
if RUN_PREFLIGHT:
    print(preflight(SPLIT_MANIFEST, BASELINE_MANIFEST, TRAIN_DIR))
else:
    print('Preflight not executed.')


## Separately approved sanity training
This cell can download the pinned model and train it. FP32, batch 2, accumulation 4, AdamW, at most 200 optimizer steps. Save checkpoints and diagnose generated training exact matches every 50 steps. All 893 validation images remain excluded. At least 31/32 is a memorization gate, not generalization accuracy. It never authorizes larger training automatically.
Nonfinite loss/gradients, CUDA OOM, interruption, or runtime errors stop the experiment. Keep completed checkpoints and partial logs. Do not silently resume or reuse output directories. A later one-epoch pilot must start from the original pretrained revision, not this memorized checkpoint.

In [ ]:
if RUN_TRAINING:
    if not CLOUD_PRIVACY_AND_UPLOAD_APPROVED:
        raise RuntimeError('Separate cloud privacy/upload/training approval required.')
    report = train_sanity(SPLIT_MANIFEST, BASELINE_MANIFEST, TRAIN_DIR, OUTPUT_DIR, TrainingConfig())
    print(report)
else:
    print('Training disabled; no model loaded by this cell.')


## Review and export after approval
Review finite loss progression and the generated-training memorization gate. Investigate failed convergence; do not automatically increase steps or train the full partition. Validation evaluation is a separate future operation using the frozen 100 images.
Download completed checkpoint directories, run_config.json, training_log.jsonl, and run_status.json. Verify checksums.json locally. Checkpoints include safetensors model files, processor/tokenizer files, config and provenance. optimizer.pt is a trusted-local PyTorch artifact: do not load untrusted pickle files. Automatic resume and exact RNG restoration are not implemented. Do not publish crops, raw reference labels, or individual generated transcriptions.